In [ ]:
-- Run this cell to ensure a dedicated monitoring schema and the historical audit logging table exist.

CREATE SCHEMA IF NOT EXISTS schema_monitoring;
USE SCHEMA schema_monitoring;

CREATE TABLE IF NOT EXISTS schema_snapshots (
    snapshot_timestamp TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
    table_catalog VARCHAR,
    table_schema VARCHAR,
    table_name VARCHAR,
    column_name VARCHAR,
    data_type VARCHAR,
    ordinal_position NUMBER,
    is_nullable VARCHAR
);

CREATE TABLE IF NOT EXISTS detected_schema_drift (
    detection_timestamp TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
    table_schema VARCHAR,
    table_name VARCHAR,
    column_name VARCHAR,
    change_category VARCHAR, -- COLUMN_ADDED, COLUMN_DROPPED, DATA_TYPE_CHANGED
    details VARCHAR
);


In [ ]:
-- This cell collects the active schema state directly from Snowflake's `INFORMATION_SCHEMA`. 
-- Customize the `WHERE` clause to target specific databases or schemas you want to monitor.

CREATE OR REPLACE TEMPORARY TABLE current_schema_state AS
SELECT 
    CURRENT_TIMESTAMP() as snapshot_timestamp,
    table_catalog, table_schema, table_name, column_name, data_type, ordinal_position, is_nullable
FROM INFORMATION_SCHEMA.COLUMNS
WHERE table_schema NOT IN ('INFORMATION_SCHEMA', 'SCHEMA_MONITORING') 
  AND table_catalog = CURRENT_DATABASE();


In [ ]:
-- If this is the first execution, this section will return no records. On subsequent runs, it accurately flags structural changes.

CREATE OR REPLACE TEMPORARY TABLE detected_drift_summary AS
WITH previous_baseline AS (
    SELECT * FROM schema_snapshots
    QUALIFY ROW_NUMBER() OVER (PARTITION BY table_schema, table_name, column_name ORDER BY snapshot_timestamp DESC) = 1
)
-- 1. Added columns
SELECT cur.table_schema, cur.table_name, cur.column_name, 'COLUMN_ADDED' AS change_category,
       'Column ' || cur.column_name || ' was added to ' || cur.table_schema || '.' || cur.table_name || ' with type ' || cur.data_type AS details
FROM current_schema_state cur
LEFT JOIN previous_baseline pre ON cur.table_schema = pre.table_schema AND cur.table_name = pre.table_name AND cur.column_name = pre.column_name
WHERE pre.column_name IS NULL
UNION ALL
-- 2. Dropped columns
SELECT pre.table_schema, pre.table_name, pre.column_name, 'COLUMN_DROPPED' AS change_category,
       'Column ' || pre.column_name || ' was dropped from ' || pre.table_schema || '.' || pre.table_name AS details
FROM previous_baseline pre
LEFT JOIN current_schema_state cur ON pre.table_schema = cur.table_schema AND pre.table_name = cur.table_name AND pre.column_name = cur.column_name
WHERE cur.column_name IS NULL
UNION ALL
-- 3. Modified Datatypes
SELECT cur.table_schema, cur.table_name, cur.column_name, 'DATA_TYPE_CHANGED' AS change_category,
       'Column ' || cur.column_name || ' type changed from ' || pre.data_type || ' to ' || cur.data_type AS details
FROM current_schema_state cur
JOIN previous_baseline pre ON cur.table_schema = pre.table_schema AND cur.table_name = pre.table_name AND cur.column_name = pre.column_name
WHERE cur.data_type != pre.data_type;

SELECT * FROM detected_drift_summary;


In [ ]:
-- This cell saves the fresh baseline record and moves any detected drift items into your persistent tracking table.

INSERT INTO detected_schema_drift (table_schema, table_name, column_name, change_category, details)
SELECT table_schema, table_name, column_name, change_category, details FROM detected_drift_summary;

INSERT INTO schema_snapshots (snapshot_timestamp, table_catalog, table_schema, table_name, column_name, data_type, ordinal_position, is_nullable)
SELECT snapshot_timestamp, table_catalog, table_schema, table_name, column_name, data_type, ordinal_position, is_nullable FROM current_schema_state;
